This notebook handles:

Multiple models

Evaluation

Selection

MLflow tracking

Saving best model ONLY

In [15]:
import pandas as pd
import numpy as np
import os
import joblib

import mlflow
import mlflow.sklearn

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

MLflow Setup

In [16]:
mlflow.set_tracking_uri("file:./mlruns")

mlflow.set_experiment("music_recommendation_mlops")

<Experiment: artifact_location='file:c:/Users/god/Desktop/music_recomendations_system/notebooks/mlruns/277328906693477146', creation_time=1775637212959, experiment_id='277328906693477146', last_update_time=1775637212959, lifecycle_stage='active', name='music_recommendation_mlops', tags={}, trace_location=None, workspace='default'>

Load Data

In [17]:
content_df = pd.read_csv("../data/processed/content_featured_data.csv")

collab_df = pd.read_csv("../data/raw/collaborative_filtering_data.csv")

CONTENT-BASED MODELS

TF-IDF Model

In [18]:
def train_tfidf(content_df):

    text_data = content_df["Track"] + " " + content_df["Artist"]

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(text_data)

    similarity = cosine_similarity(tfidf_matrix)

    return similarity, vectorizer

Cosine (Numerical)

In [19]:
def train_cosine_numeric(content_df):

    features = [
        'Danceability','Energy','Loudness','Speechiness',
        'Acousticness','Instrumentalness','Liveness',
        'Valence','Tempo','Popularity','mood_score','intensity'
    ]

    scaler = StandardScaler()
    scaled = scaler.fit_transform(content_df[features])

    similarity = cosine_similarity(scaled)

    return similarity, scaler

KNN Model

In [1]:
def train_knn(content_df):

    features = [
        'Danceability','Energy','Loudness','Speechiness',
        'Acousticness','Instrumentalness','Liveness',
        'Valence','Tempo','Popularity','mood_score','intensity'
    ]

    scaler = StandardScaler()
    scaled = scaler.fit_transform(content_df[features])

    knn = NearestNeighbors(metric="cosine")
    knn.fit(scaled)

    return knn, scaler

COLLABORATIVE MODELS

User-Item Matrix

In [21]:
def create_matrix(collab_df):

    matrix = collab_df.pivot_table(
        index='User_ID',
        columns='Artist',
        values='Play_Count',
        fill_value=0
    )

    return matrix

User-User Similarity

In [22]:
def train_user_user(matrix):

    similarity = cosine_similarity(matrix)

    return similarity

Item-Item Similarity

In [23]:
def train_item_item(matrix):

    similarity = cosine_similarity(matrix.T)

    return similarity

SVD Model

In [24]:
def train_svd(matrix):

    svd = TruncatedSVD(n_components=20)

    latent = svd.fit_transform(matrix)

    return svd, latent

EVALUATION

In [29]:
def evaluate_similarity_matrix(similarity):

    # higher average similarity → better clustering of similar songs
    score = similarity.mean()

    return score


def evaluate_knn(knn, scaled):

    distances, _ = knn.kneighbors(scaled)

    score = 1 - distances.mean()

    return score


def evaluate_svd(latent):

    score = np.var(latent)

    return score

Train All Models + MLflow

In [30]:
results = {}

matrix = create_matrix(collab_df)

# =====================
# CONTENT MODELS
# =====================

with mlflow.start_run(run_name="content_tfidf"):

    sim, vec = train_tfidf(content_df)

    score = evaluate_similarity_matrix(sim)

    results["content_tfidf"] = score

    mlflow.log_param("model_type", "content")
    mlflow.log_param("algorithm", "TF-IDF")
    mlflow.log_metric("score", score)


with mlflow.start_run(run_name="content_cosine"):

    sim, scaler = train_cosine_numeric(content_df)

    score = evaluate_similarity_matrix(sim)

    results["content_cosine"] = score

    mlflow.log_param("model_type", "content")
    mlflow.log_param("algorithm", "cosine_numeric")
    mlflow.log_metric("score", score)


with mlflow.start_run(run_name="content_knn"):

    knn, scaler = train_knn(content_df)

    scaled = scaler.transform(content_df[
        ['Danceability','Energy','Loudness','Speechiness',
        'Acousticness','Instrumentalness','Liveness',
        'Valence','Tempo','Popularity','mood_score','intensity']
    ])

    score = evaluate_knn(knn, scaled)

    results["content_knn"] = score

    mlflow.log_param("model_type", "content")
    mlflow.log_param("algorithm", "knn")
    mlflow.log_metric("score", score)


# =====================
# COLLAB MODELS
# =====================

with mlflow.start_run(run_name="collab_user_user"):

    sim = train_user_user(matrix)

    score = evaluate_similarity_matrix(sim)

    results["collab_user_user"] = score

    mlflow.log_param("model_type", "collaborative")
    mlflow.log_param("algorithm", "user-user")
    mlflow.log_metric("score", score)


with mlflow.start_run(run_name="collab_item_item"):

    sim = train_item_item(matrix)

    score = evaluate_similarity_matrix(sim)

    results["collab_item_item"] = score

    mlflow.log_param("model_type", "collaborative")
    mlflow.log_param("algorithm", "item-item")
    mlflow.log_metric("score", score)


with mlflow.start_run(run_name="collab_svd"):

    svd, latent = train_svd(matrix)

    score = evaluate_svd(latent)

    results["collab_svd"] = score

    mlflow.log_param("model_type", "collaborative")
    mlflow.log_param("algorithm", "svd")
    mlflow.log_metric("score", score)

In [31]:
content_results = {
    k:v for k,v in results.items()
    if "content" in k
}

collab_results = {
    k:v for k,v in results.items()
    if "collab" in k
}

best_content_model = max(content_results, key=content_results.get)

best_collab_model = max(collab_results, key=collab_results.get)

print("Best Content Model:", best_content_model)
print("Best Collaborative Model:", best_collab_model)

Best Content Model: content_knn
Best Collaborative Model: collab_svd


In [ ]:
os.makedirs("../models/content", exist_ok=True)
os.makedirs("../models/collaborative", exist_ok=True)

# ======================
# SAVE BEST CONTENT MODEL
# ======================

name = best_content_model.split("_")[1]

if name == "tfidf":

    sim, vec = train_tfidf(content_df)

    joblib.dump(sim, "../models/content/similarity.pkl")

    joblib.dump(vec, "../models/content/vectorizer.pkl")


elif name == "cosine":

    sim, scaler = train_cosine_numeric(content_df)

    joblib.dump(sim, "../models/content/similarity.pkl")

    joblib.dump(scaler, "../models/content/scaler.pkl")


elif name == "knn":

    knn, scaler = train_knn(content_df)

    joblib.dump(knn, "../models/content/model.pkl")

    joblib.dump(scaler, "../models/content/scaler.pkl")


joblib.dump(content_df, "../models/content/data.pkl")

joblib.dump(name, "../models/content/model_name.pkl")


# ======================
# SAVE BEST COLLAB MODEL
# ======================

name = best_collab_model.split("_")[1]

matrix = create_matrix(collab_df)

if name == "user_user":

    sim = train_user_user(matrix)

    joblib.dump(sim, "../models/collaborative/similarity.pkl")


elif name == "item_item":

    sim = train_item_item(matrix)

    joblib.dump(sim, "../models/collaborative/similarity.pkl")


elif name == "svd":

    svd, latent = train_svd(matrix)

    joblib.dump(svd, "../models/collaborative/model.pkl")

    joblib.dump(latent, "../models/collaborative/matrix.pkl")


joblib.dump(matrix, "../models/collaborative/matrix.pkl")

joblib.dump(name, "../models/collaborative/model_name.pkl")

['../models/collaborative/model_name.pkl']